In [2]:
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd
import requests
import mimetypes

# === 1) 改這兩個 ===
WEBHOOK_URL = "http://localhost:5678/webhook-test/carbon/upload"  # 你的 n8n Webhook URL
BASE_DIR = Path(r"C:\Users\echo6\Downloads\模擬公司-20250829T033655Z-1-001\模擬公司")              # 你的資料夾（放 CSV / XLSX）

# === 2) n8n 端希望收到的欄位名（Binary Property） ===
# 這些就是你在 n8n Spreadsheet File「Binary Property」要填的值
REQUIRED_FIELDS = ["scope1", "scope2", "scope3"]   # 最少三個
OPTIONAL_FIELDS = ["master", "company_info", "suppliers", "products", "orders", "shipments", "finance", "hr"]

# === 3) 每個欄位可能的檔名（會依序嘗試，先找到哪個就用哪個）===
CANDIDATES = {
    "scope1":       ["碳盤查_scope1.csv", "碳盤查_Scope1.xlsx"],
    "scope2":       ["碳盤查_scope2.csv", "碳盤查_Scope2.xlsx"],
    "scope3":       ["碳盤查_scope3.csv", "碳盤查_Scope3.xlsx"],
    "master":       ["碳盤查數據集_總表.csv", "碳盤查數據集_總表.xlsx"],  # 可選
    # 公司原始資料（依你需要逐步加）
    "company_info": ["company_info.csv", "公司基本資訊.xlsx", "公司主檔.xlsx"],
    "suppliers":    ["suppliers.csv", "供應商名錄.xlsx"],
    "products":     ["products.csv", "產品資料.xlsx"],
    "orders":       ["orders.csv", "訂單資料.xlsx"],
    "shipments":    ["shipments.csv", "出貨資料.xlsx"],
    "finance":      ["finance.csv", "財務資料.xlsx"],
    "hr":           ["hr.csv", "人力資源.xlsx"],
}

# === 工具函式 ===
def find(base: Path, names: list[str]) -> Path | None:
    """在 base 之下遞迴尋找第一個符合 names 任一檔名的檔案"""
    lowers = [n.lower() for n in names]
    for p in base.rglob("*"):
        if p.is_file() and p.name.lower() in lowers:
            return p
    return None

def read_to_csv_bytes(path: Path) -> bytes:
    """
    將 .csv 直接讀 bytes；.xlsx 讀第一個工作表轉 CSV（UTF-8 with BOM）
    讓 n8n 的 Spreadsheet File > File to JSON 能直接吃
    """
    suf = path.suffix.lower()
    if suf == ".csv":
        data = path.read_bytes()
        # 確保是 UTF-8（如果你知道是 Big5/CP950，可自行轉）
        try:
            _ = data.decode("utf-8")
            return data
        except UnicodeDecodeError:
            text = path.read_text(encoding="cp950", errors="ignore")
            return text.encode("utf-8-sig")
    elif suf in [".xlsx", ".xls"]:
        df = pd.read_excel(path, sheet_name=0, dtype=str, engine="openpyxl")
        return df.to_csv(index=False).encode("utf-8-sig")
    else:
        raise ValueError(f"不支援的檔案類型：{path}")

def build_multipart(form_map: dict[str, Path]) -> dict:
    """
    把 {field: Path} 轉成 requests 可用的 files 參數
    """
    files = {}
    for field, p in form_map.items():
        csv_bytes = read_to_csv_bytes(p)
        # filename：給 n8n 參考，副檔名 .csv 可避免下游猜不到
        files[field] = (f"{field}.csv", csv_bytes, "text/csv")
    return files

# === 主流程 ===
if __name__ == "__main__":
    assert BASE_DIR.exists(), f"資料夾不存在：{BASE_DIR}"

    # 1) 逐欄位找檔
    found: dict[str, Path] = {}
    missing_required = []

    for field in REQUIRED_FIELDS + OPTIONAL_FIELDS:
        p = find(BASE_DIR, CANDIDATES[field])
        if p:
            found[field] = p
            print(f"[OK] {field:12s} ← {p.name}")
        else:
            tag = "必需" if field in REQUIRED_FIELDS else "可選"
            print(f"[{tag}缺] {field:12s} 候選檔名：{', '.join(CANDIDATES[field])}")
            if field in REQUIRED_FIELDS:
                missing_required.append(field)

    if missing_required:
        raise SystemExit(f"❌ 缺少必需檔案：{', '.join(missing_required)}\n請放入對應檔名其中之一後再試。")

    # 2) 組 multipart/form-data
    files = build_multipart(found)
    print(f"\n[POST] {WEBHOOK_URL}")
    print("[Fields] " + ", ".join(files.keys()))

    # 3) 送出
    try:
        r = requests.post(WEBHOOK_URL, files=files, timeout=300)
        print("HTTP", r.status_code)
        # 回傳內容可能很長，先印前 2k 字協助除錯
        print(r.text[:2000])
        r.raise_for_status()
        print("✅ 上傳完成。")
    except requests.RequestException as e:
        print("❌ 上傳失敗：", e)


[OK] scope1       ← 碳盤查_scope1.csv
[OK] scope2       ← 碳盤查_scope2.csv
[OK] scope3       ← 碳盤查_scope3.csv
[OK] master       ← 碳盤查數據集_總表.csv
[OK] company_info ← 公司主檔.xlsx
[OK] suppliers    ← 供應商名錄.xlsx
[OK] products     ← 產品資料.xlsx
[OK] orders       ← 訂單資料.xlsx
[OK] shipments    ← 出貨資料.xlsx
[OK] finance      ← 財務資料.xlsx
[OK] hr           ← 人力資源.xlsx

[POST] http://localhost:5678/webhook-test/carbon/upload
[Fields] scope1, scope2, scope3, master, company_info, suppliers, products, orders, shipments, finance, hr
HTTP 200
{"message":"Workflow was started"}
✅ 上傳完成。
